In [ ]:
#hide
! [ -e /content ] && pip install -Uqq fastbook
import fastbook
fastbook.setup_book()

# Una rete neurale dalle fondamenta

Questo capitolo inizia un viaggio in cui scaveremo in profondità all'interno dei modelli che abbiamo utilizzato nei capitoli precedenti. Tratteremo molte delle cose che abbiamo visto prima, ma questa volta esamineremo molto più da vicino i dettagli di implementazione e molto meno attentamente le questioni pratiche su come e perché le cose sono come sono.

Costruiremo tutto da zero, utilizzando solo l'indicizzazione di base in un tensore. Scriveremo una rete neurale da zero, quindi implementeremo manualmente la backpropagation, in modo da sapere esattamente cosa sta succedendo in PyTorch quando chiamiamo "loss.backward". Vedremo anche come estendere PyTorch con funzioni *autograd* personalizzate che ci consentono di specificare i nostri calcoli in avanti e all'indietro.

## Costruire uno strato di rete neurale da zero

Iniziamo rinfrescando la nostra comprensione di come viene utilizzata la moltiplicazione di matrici in una rete neurale di base. Dato che stiamo costruendo tutto da zero, inizialmente non utilizzeremo altro che semplice Python (ad eccezione dell'indicizzazione nei tensori PyTorch), quindi sostituiremo il semplice Python con la funzionalità PyTorch una volta che avremo visto come crearlo.

### Modellazione di un neurone

Un neurone riceve un dato numero di input e ha un peso interno per ciascuno di essi. Somma gli input ponderati per produrre un output e aggiunge un bias interno. In matematica, questo può essere scritto come:

$$ fuori = \sum_{i=1}^{n} x_{i} w_{i} + b$$

se chiamiamo i nostri input $(x_{1},\dots,x_{n})$, i nostri pesi $(w_{1},\dots,w_{n})$ e il nostro bias $b$. Nel codice questo si traduce in:

```pitone
output = sum([x*w for x,w in zip(input,weights)]) + bias
```

Questo output viene quindi immesso in una funzione non lineare chiamata *funzione di attivazione* prima di essere inviato a un altro neurone. Nel deep learning il più comune di questi è l'*unità lineare rettificata*, o *ReLU*, che, come abbiamo visto, è un modo elegante per dire:
```pitone
def relu(x): restituisce x se x >= 0 altrimenti 0
```

Viene quindi costruito un modello di deep learning impilando molti di questi neuroni in strati successivi. Creiamo un primo strato con un certo numero di neuroni (noto come *dimensione nascosta*) e colleghiamo tutti gli input a ciascuno di quei neuroni. Tale strato è spesso chiamato *strato completamente connesso* o *strato denso* (per densamente connesso), o *strato lineare*. 

Richiede di calcolare, per ogni `input` nel nostro batch e ciascun neurone con un dato `peso`, il prodotto scalare:

```pitone
sum([x*w for x,w in zip(input,peso)])
```

Se hai studiato un po' di algebra lineare, potresti ricordare che molti di questi prodotti scalari si verificano quando esegui una *moltiplicazione di matrici*. Più precisamente, se i nostri input sono in una matrice "x" con una dimensione di "batch_size" per "n_inputs" e se abbiamo raggruppato i pesi dei nostri neuroni in una matrice "w" di dimensione "n_neurons" per "n_inputs" (ogni neurone deve avere lo stesso numero di pesi quanti sono gli input) e tutti i bias in un vettore "b" di dimensione "n_neurons", allora l'output di questo livello completamente connesso è:

```pitone
y = x @ w.t() + b
```

dove "@" rappresenta il prodotto della matrice e "w.t()" è la matrice trasposta di "w". L'output "y" ha quindi la dimensione "batch_size" per "n_neurons" e nella posizione "(i,j)" abbiamo (per i matematici là fuori):

$$y_{i,j} = \sum_{k=1}^{n} x_{i,k} w_{k,j} + b_{j}$$

Oppure in codice:

```pitone
y[i,j] = somma([a * b for a,b in zip(x[i,:],w[j,:])]) + b[j]
```

La trasposizione è necessaria perché nella definizione matematica del prodotto di matrice `m @ n`, il coefficiente `(i,j)` è:

```pitone
sum([a * b for a,b in zip(m[i,:],n[:,j])])
```

Quindi l'operazione di base di cui abbiamo bisogno è una moltiplicazione di matrici, poiché è ciò che è nascosto nel nucleo di una rete neurale.

### Moltiplicazione di matrici da zero

Scriviamo una funzione che calcola il prodotto matrice di due tensori, prima di permetterci di usarne la versione PyTorch. Utilizzeremo solo l'indicizzazione nei tensori PyTorch:

In [ ]:
import torch
from torch import tensor

Avremo bisogno di tre cicli `for` nidificati: uno per gli indici di riga, uno per gli indici di colonna e uno per la somma interna. "ac" e "ar" rappresentano rispettivamente il numero di colonne di "a" e il numero di righe di "a" (la stessa convenzione è seguita per "b"), e ci assicuriamo che sia possibile calcolare il prodotto della matrice controllando che "a" abbia tante colonne quante sono le righe di "b":

In [ ]:
def matmul(a,b):
    ar,ac = a.shape # n_rows * n_cols
    br,bc = b.shape
    assert ac==br
    c = torch.zeros(ar, bc)
    for i in range(ar):
        for j in range(bc):
            for k in range(ac): c[i,j] += a[i,k] * b[k,j]
    return c

Per verificarlo, faremo finta (usando matrici casuali) di lavorare con un piccolo lotto di 5 immagini MNIST, appiattite in vettori 28×28, con un modello lineare per trasformarle in 10 attivazioni:

In [ ]:
m1 = torch.randn(5,28*28)
m2 = torch.randn(784,10)

Cronometrizziamo la nostra funzione, utilizzando il comando "magico" di Jupyter `%time`:

In [ ]:
%time t1=matmul(m1, m2)

CPU times: user 1.15 s, sys: 4.09 ms, total: 1.15 s
Wall time: 1.15 s


E guarda come si confronta con `@` integrato di PyTorch:

In [ ]:
%timeit -n 20 t2=m1@m2

14 µs ± 8.95 µs per loop (mean ± std. dev. of 7 runs, 20 loops each)


Come possiamo vedere, in Python tre cicli annidati sono una pessima idea! Python è un linguaggio lento e non sarà molto efficiente. Qui vediamo che PyTorch è circa 100.000 volte più veloce di Python, e questo prima ancora di iniziare a utilizzare la GPU!

Da dove viene questa differenza? PyTorch non ha scritto la moltiplicazione della matrice in Python, ma piuttosto in C++ per renderla veloce. In generale, ogni volta che eseguiamo calcoli sui tensori dovremo *vettorizzarli* in modo da poter sfruttare la velocità di PyTorch, solitamente utilizzando due tecniche: aritmetica elementare e broadcasting.

### Aritmetica elementare

Tutti gli operatori di base (`+`, `-`, `*`, `/`, `>`, `<`, `==`) possono essere applicati per elemento. Ciò significa che se scriviamo `a+b` per due tensori `a` e `b` che hanno la stessa forma, otterremo un tensore composto dalle somme degli elementi di `a` e `b`:

In [ ]:
a = tensor([10., 6, -4])
b = tensor([2., 8, 7])
a + b

tensor([12., 14.,  3.])

Gli operatori booleani restituiranno un array di booleani:

In [ ]:
a < b

tensor([False,  True,  True])

Se vogliamo sapere se ogni elemento di "a" è minore dell'elemento corrispondente in "b" o se due tensori sono uguali, dobbiamo combinare queste operazioni elementarie con "torch.all":

In [ ]:
(a < b).all(), (a==b).all()

(tensor(False), tensor(False))

Operazioni di riduzione come `all()`, `sum()` e `mean()` restituiscono tensori con un solo elemento, chiamati tensori di rango 0. Se vuoi convertirlo in un semplice booleano o numero Python, devi chiamare `.item()`:

In [ ]:
(a + b).mean().item()

9.666666984558105

Le operazioni elementarie funzionano su tensori di qualsiasi rango, purché abbiano la stessa forma:

In [ ]:
m = tensor([[1., 2, 3], [4,5,6], [7,8,9]])
m*m

tensor([[ 1.,  4.,  9.],
        [16., 25., 36.],
        [49., 64., 81.]])

Tuttavia non è possibile eseguire operazioni elementari su tensori che non hanno la stessa forma (a meno che non siano trasmettibili, come discusso nella sezione successiva):

In [ ]:
n = tensor([[1., 2, 3], [4,5,6]])
m*n

RuntimeError: The size of tensor a (3) must match the size of tensor b (2) at non-singleton dimension 0

Con l'aritmetica elementare, possiamo rimuovere uno dei nostri tre cicli nidificati: possiamo moltiplicare i tensori che corrispondono alla `i`-esima riga di `a` e alla `j`-esima colonna di `b` prima di sommare tutti gli elementi, il che accelererà le cose perché il ciclo interno verrà ora eseguito da PyTorch alla velocità C. 

Per accedere a una colonna o riga, possiamo semplicemente scrivere `a[i,:]` o `b[:,j]`. Il `:` significa prendere tutto in quella dimensione. Potremmo limitare questo e prendere solo una porzione di quella particolare dimensione passando un intervallo, come "1:5", invece del solo ":". In tal caso, prenderemo gli elementi nelle colonne o nelle righe da 1 a 4 (il secondo numero non è inclusivo). 

Una semplificazione è che possiamo sempre omettere i due punti finali, quindi "a[i,:]" può essere abbreviato in "a[i]". Tenendo presente tutto ciò, possiamo scrivere una nuova versione della nostra moltiplicazione di matrici:

In [ ]:
def matmul(a,b):
    ar,ac = a.shape
    br,bc = b.shape
    assert ac==br
    c = torch.zeros(ar, bc)
    for i in range(ar):
        for j in range(bc): c[i,j] = (a[i] * b[:,j]).sum()
    return c

In [ ]:
%timeit -n 20 t3 = matmul(m1,m2)

1.7 ms ± 88.1 µs per loop (mean ± std. dev. of 7 runs, 20 loops each)


Siamo già ~700 volte più veloci, semplicemente rimuovendo il ciclo "for" interno! E questo è solo l'inizio: con la trasmissione possiamo eliminare un altro loop e ottenere una velocità ancora più importante.

### Trasmissione

Come abbiamo discusso in <<chapter_mnist_basics>>, broadcasting è un termine introdotto dalla [libreria NumPy](https://docs.scipy.org/doc/) che descrive come vengono trattati tensori di ranghi diversi durante le operazioni aritmetiche. Ad esempio, è ovvio che non c'è modo di aggiungere una matrice 3×3 con una matrice 4×5, ma cosa succede se vogliamo aggiungere uno scalare (che può essere rappresentato come un tensore 1×1) con una matrice? Oppure un vettore di dimensione 3 con matrice 3×4? In entrambi i casi possiamo trovare un modo per dare un senso a questa operazione.

La trasmissione fornisce regole specifiche per codificare quando le forme sono compatibili quando si tenta di eseguire un'operazione basata sugli elementi e come il tensore della forma più piccola viene espanso per corrispondere al tensore della forma più grande. È essenziale padroneggiare queste regole se vuoi essere in grado di scrivere codice che venga eseguito rapidamente. In questa sezione espanderemo la nostra precedente trattazione della trasmissione per comprendere queste regole.

#### Trasmissione con uno scalare

La trasmissione con uno scalare è il tipo di trasmissione più semplice. Quando abbiamo un tensore "a" e uno scalare, immaginiamo semplicemente un tensore della stessa forma di "a" riempito con quello scalare ed eseguiamo l'operazione:

In [ ]:
a = tensor([10., 6, -4])
a > 0

tensor([ True,  True, False])

Come possiamo fare questo confronto? "0" viene *trasmesso* per avere le stesse dimensioni di "a". Si noti che ciò viene fatto senza creare un tensore pieno di zeri in memoria (sarebbe molto inefficiente). 

Questo è molto utile se vuoi normalizzare il tuo set di dati sottraendo la media (uno scalare) dall'intero set di dati (una matrice) e dividendo per la deviazione standard (un altro scalare):

In [ ]:
m = tensor([[1., 2, 3], [4,5,6], [7,8,9]])
(m - 5) / 2.73

tensor([[-1.4652, -1.0989, -0.7326],
        [-0.3663,  0.0000,  0.3663],
        [ 0.7326,  1.0989,  1.4652]])

E se avessimo medie diverse per ogni riga della matrice? in tal caso dovrai trasmettere un vettore a una matrice.

#### Trasmissione di un vettore a una matrice

Possiamo trasmettere un vettore a una matrice come segue:

In [ ]:
c = tensor([10.,20,30])
m = tensor([[1., 2, 3], [4,5,6], [7,8,9]])
m.shape,c.shape

(torch.Size([3, 3]), torch.Size([3]))

In [ ]:
m + c

tensor([[11., 22., 33.],
        [14., 25., 36.],
        [17., 28., 39.]])

Qui gli elementi di `c` vengono espansi per creare tre righe corrispondenti, rendendo possibile l'operazione. Ancora una volta, PyTorch in realtà non crea tre copie di "c" in memoria. Questo viene fatto dietro le quinte con il metodo "expand_as":

In [ ]:
c.expand_as(m)

tensor([[10., 20., 30.],
        [10., 20., 30.],
        [10., 20., 30.]])

Se osserviamo il tensore corrispondente, possiamo chiedere la sua proprietà `storage` (che mostra il contenuto effettivo della memoria utilizzata per il tensore) per verificare che non siano memorizzati dati inutili:

In [ ]:
t = c.expand_as(m)
t.storage()

 10.0
 20.0
 30.0
[torch.FloatStorage of size 3]

Anche se il tensore ha ufficialmente nove elementi, in memoria vengono archiviati solo tre scalari. Ciò è possibile grazie al trucco intelligente di dare a quella dimensione un *stride* pari a 0 (il che significa che quando PyTorch cerca la riga successiva aggiungendo il passo, non si muove):

In [ ]:
t.stride(), t.shape

((0, 1), torch.Size([3, 3]))

Poiché `m` ha dimensioni 3×3, ci sono due modi per eseguire la trasmissione. Il fatto che sia stato fatto nell'ultima dimensione è una convenzione che deriva dalle regole della radiodiffusione e non ha nulla a che fare con il modo in cui abbiamo ordinato i nostri tensori. Se invece facciamo così, otteniamo lo stesso risultato:

In [ ]:
c + m

tensor([[11., 22., 33.],
        [14., 25., 36.],
        [17., 28., 39.]])

Infatti, è possibile trasmettere solo un vettore di dimensione `n` con una matrice di dimensione `m` per `n`:

In [ ]:
c = tensor([10.,20,30])
m = tensor([[1., 2, 3], [4,5,6]])
c+m

tensor([[11., 22., 33.],
        [14., 25., 36.]])

Questo non funzionerà:

In [ ]:
c = tensor([10.,20])
m = tensor([[1., 2, 3], [4,5,6]])
c+m

RuntimeError: The size of tensor a (2) must match the size of tensor b (3) at non-singleton dimension 1

Se vogliamo trasmettere nell'altra dimensione, dobbiamo cambiare la forma del nostro vettore per renderlo una matrice 3×1. Questo viene fatto con il metodo `unsqueeze` in PyTorch:

In [ ]:
c = tensor([10.,20,30])
m = tensor([[1., 2, 3], [4,5,6], [7,8,9]])
c = c.unsqueeze(1)
m.shape,c.shape

(torch.Size([3, 3]), torch.Size([3, 1]))

Questa volta, "c" viene espanso sul lato della colonna:

In [ ]:
c+m

tensor([[11., 12., 13.],
        [24., 25., 26.],
        [37., 38., 39.]])

Come prima, solo tre scalari sono archiviati in memoria:

In [ ]:
t = c.expand_as(m)
t.storage()

 10.0
 20.0
 30.0
[torch.FloatStorage of size 3]

E il tensore espanso ha la forma giusta perché la dimensione della colonna ha un passo pari a 0:

In [ ]:
t.stride(), t.shape

((1, 0), torch.Size([3, 3]))

Con la trasmissione, per impostazione predefinita, se dobbiamo aggiungere dimensioni, queste vengono aggiunte all'inizio. Durante la trasmissione precedente, Pytorch stava eseguendo `c.unsqueeze(0)` dietro le quinte:

In [ ]:
c = tensor([10.,20,30])
c.shape, c.unsqueeze(0).shape,c.unsqueeze(1).shape

(torch.Size([3]), torch.Size([1, 3]), torch.Size([3, 1]))

Il comando "unsqueeze" può essere sostituito dall'indicizzazione "None":

In [ ]:
c.shape, c[None,:].shape,c[:,None].shape

(torch.Size([3]), torch.Size([1, 3]), torch.Size([3, 1]))

Puoi sempre omettere i due punti finali e "..." indica tutte le dimensioni precedenti:

In [ ]:
c[None].shape,c[...,None].shape

(torch.Size([1, 3]), torch.Size([3, 1]))

Con questo, possiamo rimuovere un altro ciclo "for" nella nostra funzione di moltiplicazione di matrici. Ora, invece di moltiplicare `a[i]` per `b[:,j]`, possiamo moltiplicare `a[i]` per l'intera matrice `b` utilizzando il broadcasting, quindi sommare i risultati:

In [ ]:
def matmul(a,b):
    ar,ac = a.shape
    br,bc = b.shape
    assert ac==br
    c = torch.zeros(ar, bc)
    for i in range(ar):
#       c[i,j] = (a[i,:]          * b[:,j]).sum() # previous
        c[i]   = (a[i  ].unsqueeze(-1) * b).sum(dim=0)
    return c

In [ ]:
%timeit -n 20 t4 = matmul(m1,m2)

357 µs ± 7.2 µs per loop (mean ± std. dev. of 7 runs, 20 loops each)


Ora siamo 3.700 volte più veloci della nostra prima implementazione! Prima di proseguire, discutiamo un po' più in dettaglio le regole della trasmissione.

#### Regole di trasmissione

Quando si opera su due tensori, PyTorch confronta le loro forme in base agli elementi. Inizia con le *dimensioni finali* e procede all'indietro, aggiungendo 1 quando incontra dimensioni vuote. Due dimensioni sono *compatibili* quando si verifica una delle seguenti condizioni:

- Sono uguali.
- Uno di questi è 1, nel qual caso quella dimensione viene trasmessa per renderla uguale all'altra.

Non è necessario che gli array abbiano lo stesso numero di dimensioni. Ad esempio, se disponi di una matrice 256×256×3 di valori RGB e desideri ridimensionare ciascun colore nell'immagine in base a un valore diverso, puoi moltiplicare l'immagine per una matrice unidimensionale con tre valori. Allineando le dimensioni degli assi finali di questi array secondo le regole di trasmissione, mostra che sono compatibili:

```
Immagine (tensore 3D): 256 x 256 x 3
Scala (tensore 1d): (1) (1) 3
Risultato (tensore 3d): 256 x 256 x 3
```
    
Tuttavia, un tensore 2D di dimensione 256×256 non è compatibile con la nostra immagine:

```
Immagine (tensore 3D): 256 x 256 x 3
Scala (tensore 2d): (1) 256 x 256
Errore
```

Nei nostri esempi precedenti con una matrice 3×3 e un vettore di dimensione 3, la trasmissione veniva eseguita sulle righe:

```
Matrice (tensore 2d): 3 x 3
Vettore (tensore 1d): (1) 3
Risultato (tensore 2d): 3 x 3
```

Come esercizio, prova a determinare quali dimensioni aggiungere (e dove) quando devi normalizzare un batch di immagini di dimensione `64 x 3 x 256 x 256` con vettori di tre elementi (uno per la media e uno per la deviazione standard).

Un altro modo utile per semplificare le manipolazioni dei tensori è l'uso della convenzione delle somme di Einstein.

### Somma di Einstein

Prima di utilizzare l'operazione PyTorch `@` o `torch.matmul`, esiste un ultimo modo per implementare la moltiplicazione di matrici: la somma di Einstein (`einsum`). Questa è una rappresentazione compatta per combinare prodotti e somme in modo generale. Scriviamo un'equazione come questa:

```
ik,kj -> ij
```

Il lato sinistro rappresenta le dimensioni degli operandi, separati da virgole. Qui abbiamo due tensori che hanno ciascuno due dimensioni (`i,k` e `k,j`).  Il lato destro rappresenta le dimensioni del risultato, quindi qui abbiamo un tensore con due dimensioni `i,j`. 

Le regole della notazione delle sommatorie di Einstein sono le seguenti:

1. Gli indici ripetuti sul lato sinistro vengono sommati implicitamente se non si trovano sul lato destro.
2. Ciascun indice può apparire al massimo due volte sul lato sinistro.
3. Gli indici non ripetuti sul lato sinistro devono apparire sul lato destro.

Quindi nel nostro esempio, poiché "k" viene ripetuto, sommiamo quell'indice. Alla fine la formula rappresenta la matrice ottenuta quando mettiamo in `(i,j)` la somma di tutti i coefficienti `(i,k)` del primo tensore moltiplicati per i coefficienti `(k,j)` del secondo tensore... che è il prodotto della matrice! Ecco come possiamo codificarlo in PyTorch:

In [ ]:
def matmul(a,b): return torch.einsum('ik,kj->ij', a, b)

La somma di Einstein è un modo molto pratico di esprimere operazioni che coinvolgono l'indicizzazione e la somma di prodotti. Tieni presente che puoi avere un solo membro sul lato sinistro. Ad esempio, questo:

```pitone
torcia.einsum('ij->ji', a)
```

restituisce la trasposizione della matrice "a". Puoi anche avere tre o più membri. questo:

```pitone
torcia.einsum('bi,ij,bj->b', a, b, c)
```

restituirà un vettore di dimensione "b" dove la coordinata "k"-esima è la somma di "a[k,i] b[i,j] c[k,j]". Questa notazione è particolarmente utile quando si hanno più dimensioni a causa dei batch. Ad esempio, se hai due lotti di matrici e desideri calcolare il prodotto della matrice per lotto, potresti fare quanto segue: 

```pitone
torcia.einsum('bik,bkj->bij', a, b)
```

Torniamo alla nostra nuova implementazione `matmul` utilizzando "einsum" e osserviamo la sua velocità:

In [ ]:
%timeit -n 20 t5 = matmul(m1,m2)

68.7 µs ± 4.06 µs per loop (mean ± std. dev. of 7 runs, 20 loops each)


Come puoi vedere, non solo è pratico, ma è *molto* veloce. "einsum" è spesso il modo più veloce per eseguire operazioni personalizzate in PyTorch, senza immergersi in C++ e CUDA. (Ma generalmente non è veloce quanto il codice CUDA attentamente ottimizzato, come puoi vedere dai risultati in "Moltiplicazione di matrici da zero".)

Ora che sappiamo come implementare una moltiplicazione di matrici da zero, siamo pronti a costruire la nostra rete neurale, in particolare i suoi passaggi avanti e indietro, utilizzando solo moltiplicazioni di matrici.

## I passaggi in avanti e all'indietro

Come abbiamo visto in <<chapter_mnist_basics>>, per addestrare un modello, dovremo calcolare tutti i gradienti di una data perdita rispetto ai suoi parametri, operazione nota come *passaggio all'indietro*. Il *passaggio in avanti* è il momento in cui calcoliamo l'output del modello su un dato input, in base ai prodotti della matrice. Man mano che definiamo la nostra prima rete neurale, approfondiremo anche il problema della corretta inizializzazione dei pesi, fondamentale per far iniziare correttamente l'allenamento.

### Definizione e inizializzazione di un livello

Prenderemo innanzitutto l'esempio di una rete neurale a due strati. Come abbiamo visto, un livello può essere espresso come `y = x @ w + b`, con `x` i nostri input, "y" i nostri output, "w" i pesi del livello (che è pari al numero di input per numero di neuroni se non trasponiamo come prima) e "b" è il vettore di bias:

In [ ]:
def lin(x, w, b): return x @ w + b

Possiamo sovrapporre il secondo strato al primo, ma poiché matematicamente la composizione di due operazioni lineari è un'altra operazione lineare, questo ha senso solo se mettiamo qualcosa di non lineare nel mezzo, chiamato funzione di attivazione. Come accennato all'inizio del capitolo, nelle applicazioni di deep learning la funzione di attivazione più comunemente utilizzata è una ReLU, che restituisce il massimo tra `x` e `0`. 

In realtà non addestreremo il nostro modello in questo capitolo, quindi utilizzeremo tensori casuali per i nostri input e target. Diciamo che i nostri input sono 200 vettori di dimensione 100, che raggruppiamo in un unico batch, e i nostri obiettivi sono 200 numeri in virgola mobile casuali:

In [ ]:
x = torch.randn(200, 100)
y = torch.randn(200)

Per il nostro modello a due strati avremo bisogno di due matrici di pesi e due vettori di bias. Diciamo che abbiamo una dimensione nascosta di 50 e la dimensione di output è 1 (per uno dei nostri input, l'output corrispondente è un float in questo esempio di giocattolo). Inizializziamo i pesi in modo casuale e il bias a zero:

In [ ]:
w1 = torch.randn(100,50)
b1 = torch.zeros(50)
w2 = torch.randn(50,1)
b2 = torch.zeros(1)

Quindi il risultato del nostro primo livello è semplicemente:

In [ ]:
l1 = lin(x, w1, b1)
l1.shape

torch.Size([200, 50])

Tieni presente che questa formula funziona con il nostro batch di input e restituisce un batch con stato nascosto: "l1" è una matrice di dimensione 200 (la nostra dimensione batch) per 50 (la nostra dimensione nascosta).

Tuttavia, c'è un problema con il modo in cui il nostro modello è stato inizializzato. Per capirlo, dobbiamo guardare la media e la deviazione standard (std) di "l1":

In [ ]:
l1.mean(), l1.std()

(tensor(0.0019), tensor(10.1058))

La media è vicina allo zero, il che è comprensibile poiché sia ​​la nostra matrice di input che quella dei pesi hanno medie vicine allo zero. Ma la deviazione standard, che rappresenta la distanza delle nostre attivazioni dalla media, è passata da 1 a 10. Questo è un problema davvero grosso perché riguarda un solo livello. Le moderne reti neurali possono avere centinaia di strati, quindi se ognuno di essi moltiplica la scala delle nostre attivazioni per 10, alla fine dell'ultimo strato non avremo numeri rappresentabili da un computer.

Infatti, se effettuiamo solo 50 moltiplicazioni tra `x` e matrici casuali di dimensione 100×100, avremo:

In [ ]:
x = torch.randn(200, 100)
for i in range(50): x = x @ torch.randn(100,100)
x[0:5,0:5]

tensor([[nan, nan, nan, nan, nan],
        [nan, nan, nan, nan, nan],
        [nan, nan, nan, nan, nan],
        [nan, nan, nan, nan, nan],
        [nan, nan, nan, nan, nan]])

Il risultato è "nan" ovunque. Quindi forse la scala della nostra matrice era troppo grande e abbiamo bisogno di pesi più piccoli? Ma se utilizziamo pesi troppo piccoli, avremo il problema opposto: la scala delle nostre attivazioni andrà da 1 a 0,1 e dopo 50 strati rimarremo con zeri ovunque:

In [ ]:
x = torch.randn(200, 100)
for i in range(50): x = x @ (torch.randn(100,100) * 0.01)
x[0:5,0:5]

tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])

Quindi dobbiamo ridimensionare le nostre matrici di peso esattamente nel modo giusto in modo che la deviazione standard delle nostre attivazioni rimanga su 1. Possiamo calcolare il valore esatto da utilizzare matematicamente, come illustrato da Xavier Glorot e Yoshua Bengio in ["Understanding the Difficulty of Training Deep Feedforward Neural Networks"](http://proceedings.mlr.press/v9/glorot10a/glorot10a.pdf). La scala corretta per un dato livello è $1/\sqrt{n_{in}}$, dove $n_{in}$ rappresenta il numero di input.

Nel nostro caso, se abbiamo 100 input, dovremmo scalare le nostre matrici dei pesi di 0,1:

In [ ]:
x = torch.randn(200, 100)
for i in range(50): x = x @ (torch.randn(100,100) * 0.1)
x[0:5,0:5]

tensor([[ 0.7554,  0.6167, -0.1757, -1.5662,  0.5644],
        [-0.1987,  0.6292,  0.3283, -1.1538,  0.5416],
        [ 0.6106,  0.2556, -0.0618, -0.9463,  0.4445],
        [ 0.4484,  0.7144,  0.1164, -0.8626,  0.4413],
        [ 0.3463,  0.5930,  0.3375, -0.9486,  0.5643]])

Finalmente alcuni numeri che non sono né zeri né `nan`! Nota quanto è stabile la scala delle nostre attivazioni, anche dopo quei 50 livelli falsi:

In [ ]:
x.std()

tensor(0.7042)

Se giochi un po' con il valore della scala noterai che anche una leggera variazione da 0,1 ti porterà a numeri molto piccoli o molto grandi, quindi inizializzare correttamente i pesi è estremamente importante. 

Torniamo alla nostra rete neurale. Dato che abbiamo incasinato un po' i nostri input, dobbiamo ridefinirli:

In [ ]:
x = torch.randn(200, 100)
y = torch.randn(200)

E per i nostri pesi utilizzeremo la scala giusta, nota come *inizializzazione Xavier* (o *inizializzazione Glorot*):

In [ ]:
from math import sqrt
w1 = torch.randn(100,50) / sqrt(100)
b1 = torch.zeros(50)
w2 = torch.randn(50,1) / sqrt(50)
b2 = torch.zeros(1)

Ora se calcoliamo il risultato del primo strato, possiamo verificare che la media e la deviazione standard siano sotto controllo:

In [ ]:
l1 = lin(x, w1, b1)
l1.mean(),l1.std()

(tensor(-0.0050), tensor(1.0000))

Molto bene. Ora dobbiamo passare attraverso una ReLU, quindi definiamone una. Una ReLU rimuove gli aspetti negativi e li sostituisce con zeri, che è un altro modo per dire che blocca il nostro tensore a zero:

In [ ]:
def relu(x): return x.clamp_min(0.)

Passiamo le nostre attivazioni attraverso questo:

In [ ]:
l2 = relu(l1)
l2.mean(),l2.std()

(tensor(0.3961), tensor(0.5783))

E siamo tornati al punto di partenza: la media delle nostre attivazioni è andata a 0,4 (il che è comprensibile dato che abbiamo rimosso gli aspetti negativi) e lo std è sceso a 0,58. Quindi, come prima, dopo alcuni strati probabilmente finiremo con gli zeri:

In [ ]:
x = torch.randn(200, 100)
for i in range(50): x = relu(x @ (torch.randn(100,100) * 0.1))
x[0:5,0:5]

tensor([[0.0000e+00, 1.9689e-08, 4.2820e-08, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 1.6701e-08, 4.3501e-08, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 1.0976e-08, 3.0411e-08, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 1.8457e-08, 4.9469e-08, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 1.9949e-08, 4.1643e-08, 0.0000e+00, 0.0000e+00]])

Ciò significa che la nostra inizializzazione non era corretta. Perché? All'epoca in cui Glorot e Bengio scrissero il loro articolo, l'attivazione popolare in una rete neurale era la tangente iperbolica (tanh, che è quella che usarono) e quell'inizializzazione non tiene conto della nostra ReLU. Fortunatamente, qualcun altro ha fatto i conti per noi e ha calcolato la scala giusta da utilizzare. In ["Delving Deep into Rectifiers: Surpassing Human-Level Performance"](https://arxiv.org/abs/1502.01852) (che abbiamo già visto—è l'articolo che ha introdotto ResNet), Kaiming He et al. mostriamo che dovremmo invece utilizzare la seguente scala: $\sqrt{2 / n_{in}}$, dove $n_{in}$ è il numero di input del nostro modello. Vediamo cosa ci offre:

In [ ]:
x = torch.randn(200, 100)
for i in range(50): x = relu(x @ (torch.randn(100,100) * sqrt(2/100)))
x[0:5,0:5]

tensor([[0.2871, 0.0000, 0.0000, 0.0000, 0.0026],
        [0.4546, 0.0000, 0.0000, 0.0000, 0.0015],
        [0.6178, 0.0000, 0.0000, 0.0180, 0.0079],
        [0.3333, 0.0000, 0.0000, 0.0545, 0.0000],
        [0.1940, 0.0000, 0.0000, 0.0000, 0.0096]])

Meglio così: i nostri numeri questa volta non sono tutti azzerati. Torniamo quindi alla definizione della nostra rete neurale e utilizziamo questa inizializzazione (che si chiama *Inizializzazione Kaiming* o *Inizializzazione He*):

In [ ]:
x = torch.randn(200, 100)
y = torch.randn(200)

In [ ]:
w1 = torch.randn(100,50) * sqrt(2 / 100)
b1 = torch.zeros(50)
w2 = torch.randn(50,1) * sqrt(2 / 50)
b2 = torch.zeros(1)

Diamo un'occhiata alla scala delle nostre attivazioni dopo aver attraversato il primo strato lineare e ReLU:

In [ ]:
l1 = lin(x, w1, b1)
l2 = relu(l1)
l2.mean(), l2.std()

(tensor(0.5661), tensor(0.8339))

Molto meglio! Ora che i nostri pesi sono inizializzati correttamente, possiamo definire il nostro intero modello:

In [ ]:
def model(x):
    l1 = lin(x, w1, b1)
    l2 = relu(l1)
    l3 = lin(l2, w2, b2)
    return l3

Questo è il passaggio in avanti. Ora tutto ciò che resta da fare è confrontare il nostro output con le etichette che abbiamo (numeri casuali, in questo esempio) con una funzione di perdita. In questo caso utilizzeremo l’errore quadratico medio. (È un problema giocattolo, e questa è la funzione di perdita più semplice da utilizzare per ciò che verrà dopo, calcolare i gradienti.)

L'unica sottigliezza è che i nostri risultati e obiettivi non hanno esattamente la stessa forma: dopo aver analizzato il modello, otteniamo un risultato come questo:

In [ ]:
out = model(x)
out.shape

torch.Size([200, 1])

Per eliminare questa dimensione finale 1, utilizziamo la funzione `squeeze`:

In [ ]:
def mse(output, targ): return (output.squeeze(-1) - targ).pow(2).mean()

E ora siamo pronti a calcolare la nostra perdita:

In [ ]:
loss = mse(out, y)

Questo è tutto per il passaggio in avanti: ora guardiamo le pendenze.

### Gradienti e passaggio all'indietro

Abbiamo visto che PyTorch calcola tutti i gradienti di cui abbiamo bisogno con una chiamata magica a `loss.backward`, ma esploriamo cosa sta succedendo dietro le quinte.

Ora arriva la parte in cui dobbiamo calcolare i gradienti della perdita rispetto a tutti i pesi del nostro modello, quindi tutti i float in `w1`, `b1`, `w2` e `b2`. Per questo avremo bisogno di un po' di matematica, in particolare della *regola della catena*. Questa è la regola del calcolo che guida il modo in cui possiamo calcolare la derivata di una funzione composta:

$$(g \circ f)'(x) = g'(f(x)) f'(x)$$

> j: Trovo molto difficile comprendere questa notazione, quindi mi piace pensarla come: if `y = g(u)` e `u=f(x)`; quindi `dy/dx = dy/du * du/dx`. Le due notazioni significano la stessa cosa, quindi usa quello che funziona per te.

La nostra perdita è una grande composizione di diverse funzioni: errore quadratico medio (che è a sua volta la composizione di una media e una potenza di due), il secondo strato lineare, una ReLU e il primo strato lineare. Ad esempio, se vogliamo i gradienti della perdita rispetto a "b2" e la nostra perdita è definita da:

```
perdita = mse(out,y) = mse(lin(l2, w2, b2), y)
```

La regola della catena ci dice che abbiamo:
$$\frac{\text{d} perdita}{\text{d} b_{2}} = \frac{\text{d} perdita}{\text{d} out} \times \frac{\text{d} out}{\text{d} b_{2}} = \frac{\text{d}}{\text{d} out} mse(out, y) \times \frac{\text{d}}{\text{d} b_{2}} lin(l_{2}, w_{2}, b_{2})$$

Per calcolare i gradienti della perdita rispetto a $b_{2}$, abbiamo prima bisogno dei gradienti della perdita rispetto al nostro output $out$. Lo stesso vale se vogliamo i gradienti della perdita rispetto a $w_{2}$. Quindi, per ottenere i gradienti della perdita rispetto a $b_{1}$ o $w_{1}$, avremo bisogno dei gradienti della perdita rispetto a $l_{1}$, che a sua volta richiede i gradienti della perdita rispetto a $l_{2}$, che avrà bisogno dei gradienti della perdita rispetto a $out$.

Quindi, per calcolare tutti i gradienti di cui abbiamo bisogno per l'aggiornamento, dobbiamo iniziare dall'output del modello e procedere *all'indietro*, uno strato dopo l'altro, motivo per cui questo passaggio è noto come *backpropagation*. Possiamo automatizzarlo facendo sì che ogni funzione che abbiamo implementato (`relu`, `mse`, `lin`) fornisca il suo passo indietro: ovvero, come derivare i gradienti della perdita rispetto agli input dai gradienti della perdita rispetto all'output.

Qui popoliamo questi gradienti in un attributo di ciascun tensore, un po' come fa PyTorch con `.grad`. 

I primi sono i gradienti della perdita rispetto all’output del nostro modello (che è l’input della funzione di perdita). Annulliamo lo `squeeze` che abbiamo fatto in `mse`, quindi usiamo la formula che ci dà la derivata di $x^{2}$: $2x$. La derivata della media è solo $1/n$ dove $n$ è il numero di elementi nel nostro input:

In [ ]:
def mse_grad(inp, targ): 
    # grad of loss with respect to output of previous layer
    inp.g = 2. * (inp.squeeze() - targ).unsqueeze(-1) / inp.shape[0]

Per i gradienti di ReLU e del nostro livello lineare, utilizziamo i gradienti della perdita rispetto all'output (in `out.g`) e applichiamo la regola della catena per calcolare i gradienti della perdita rispetto all'input (in `inp.g`). La regola della catena ci dice che `inp.g = relu'(inp) * out.g`. La derivata di "relu" è 0 (quando gli input sono negativi) o 1 (quando gli input sono positivi), quindi questo ci dà:

In [ ]:
def relu_grad(inp, out):
    # grad of relu with respect to input activations
    inp.g = (inp>0).float() * out.g

Lo schema è lo stesso per calcolare i gradienti della perdita rispetto agli input, ai pesi e alla distorsione nello strato lineare:

In [ ]:
def lin_grad(inp, out, w, b):
    # grad of matmul with respect to input
    inp.g = out.g @ w.t()
    w.g = inp.t() @ out.g
    b.g = out.g.sum(0)

Non ci soffermeremo sulle formule matematiche che li definiscono poiché non sono importanti per i nostri scopi, ma dai un'occhiata alle eccellenti lezioni di calcolo della Khan Academy se sei interessato a questo argomento.

### Barra laterale: SymPy

SymPy è una libreria per il calcolo simbolico che è una libreria estremamente utile quando si lavora con il calcolo infinitesimale. Secondo la [documentazione](https://docs.sympy.org/latest/tutorial/intro.html):

> : Il calcolo simbolico si occupa del calcolo simbolico di oggetti matematici. Ciò significa che gli oggetti matematici vengono rappresentati esattamente, non approssimativamente, e le espressioni matematiche con variabili non valutate vengono lasciate in forma simbolica.

Per eseguire il calcolo simbolico, definiamo prima un *simbolo*, quindi eseguiamo un calcolo, in questo modo:

In [ ]:
from sympy import symbols,diff
sx,sy = symbols('sx sy')
diff(sx**2, sx)

2*sx

Qui SymPy ha preso per noi la derivata di `x**2`! Può ricavare la derivata di espressioni composte complicate, semplificare e fattorizzare equazioni e molto altro ancora. Al giorno d'oggi non c'è davvero molto motivo per cui qualcuno debba fare calcoli manualmente: per calcolare i gradienti, PyTorch lo fa per noi e per mostrare le equazioni, SymPy lo fa per noi!

### Fine della barra laterale

Una volta definite tali funzioni, possiamo usarle per scrivere il passaggio all'indietro. Poiché ogni gradiente viene popolato automaticamente nel tensore giusto, non abbiamo bisogno di memorizzare i risultati di quelle funzioni `_grad` da nessuna parte: dobbiamo solo eseguirle nell'ordine inverso del passaggio in avanti, per assicurarci che in ogni funzione esista `out.g`:

In [ ]:
def forward_and_backward(inp, targ):
    # forward pass:
    l1 = inp @ w1 + b1
    l2 = relu(l1)
    out = l2 @ w2 + b2
    # we don't actually need the loss in backward!
    loss = mse(out, targ)
    
    # backward pass:
    mse_grad(out, targ)
    lin_grad(l2, out, w2, b2)
    relu_grad(l1, l2)
    lin_grad(inp, l1, w1, b1)

E ora possiamo accedere ai gradienti dei parametri del nostro modello in `w1.g`, `b1.g`, `w2.g` e `b2.g`.

Abbiamo definito con successo il nostro modello: ora rendiamolo un po' più simile a un modulo PyTorch.

### Refactoring del modello

Le tre funzioni che abbiamo utilizzato hanno due funzioni associate: un passaggio in avanti e un passaggio all'indietro. Invece di scriverli separatamente, possiamo creare una classe per racchiuderli insieme. Quella classe può anche memorizzare gli input e gli output per il passaggio all'indietro. In questo modo dovremo semplicemente chiamare `backward`:

In [ ]:
class Relu():
    def __call__(self, inp):
        self.inp = inp
        self.out = inp.clamp_min(0.)
        return self.out
    
    def backward(self): self.inp.g = (self.inp>0).float() * self.out.g

`__call__` è un nome magico in Python che renderà richiamabile la nostra classe. Questo è ciò che verrà eseguito quando digitiamo `y = Relu()(x)`. Possiamo fare lo stesso per il nostro strato lineare e la perdita MSE:

In [ ]:
class Lin():
    def __init__(self, w, b): self.w,self.b = w,b
        
    def __call__(self, inp):
        self.inp = inp
        self.out = inp@self.w + self.b
        return self.out
    
    def backward(self):
        self.inp.g = self.out.g @ self.w.t()
        self.w.g = self.inp.t() @ self.out.g
        self.b.g = self.out.g.sum(0)

In [ ]:
class Mse():
    def __call__(self, inp, targ):
        self.inp = inp
        self.targ = targ
        self.out = (inp.squeeze() - targ).pow(2).mean()
        return self.out
    
    def backward(self):
        x = (self.inp.squeeze()-self.targ).unsqueeze(-1)
        self.inp.g = 2.*x/self.targ.shape[0]

Quindi possiamo inserire tutto in un modello che iniziamo con i nostri tensori `w1`, `b1`, `w2`, `b2`:

In [ ]:
class Model():
    def __init__(self, w1, b1, w2, b2):
        self.layers = [Lin(w1,b1), Relu(), Lin(w2,b2)]
        self.loss = Mse()
        
    def __call__(self, x, targ):
        for l in self.layers: x = l(x)
        return self.loss(x, targ)
    
    def backward(self):
        self.loss.backward()
        for l in reversed(self.layers): l.backward()

La cosa veramente bella di questo refactoring e della registrazione delle cose come livelli del nostro modello è che i passaggi avanti e indietro ora sono davvero facili da scrivere. Se vogliamo istanziare il nostro modello non ci resta che scrivere:

In [ ]:
model = Model(w1, b1, w2, b2)

Il passaggio in avanti può quindi essere eseguito con:

In [ ]:
loss = model(x, y)

E il passaggio all'indietro con:

In [ ]:
model.backward()

### Andiamo a PyTorch

Le classi "Lin", "Mse" e "Relu" che abbiamo scritto hanno molto in comune, quindi potremmo farle ereditare tutte dalla stessa classe base:

In [ ]:
class LayerFunction():
    def __call__(self, *args):
        self.args = args
        self.out = self.forward(*args)
        return self.out
    
    def forward(self):  raise Exception('not implemented')
    def bwd(self):      raise Exception('not implemented')
    def backward(self): self.bwd(self.out, *self.args)

Quindi dobbiamo solo implementare `forward` e `bwd` in ciascuna delle nostre sottoclassi:

In [ ]:
class Relu(LayerFunction):
    def forward(self, inp): return inp.clamp_min(0.)
    def bwd(self, out, inp): inp.g = (inp>0).float() * out.g

In [ ]:
class Lin(LayerFunction):
    def __init__(self, w, b): self.w,self.b = w,b
        
    def forward(self, inp): return inp@self.w + self.b
    
    def bwd(self, out, inp):
        inp.g = out.g @ self.w.t()
        self.w.g = inp.t() @ self.out.g
        self.b.g = out.g.sum(0)

In [ ]:
class Mse(LayerFunction):
    def forward (self, inp, targ): return (inp.squeeze() - targ).pow(2).mean()
    def bwd(self, out, inp, targ): 
        inp.g = 2*(inp.squeeze()-targ).unsqueeze(-1) / targ.shape[0]

Il resto del nostro modello può essere lo stesso di prima. Questo si sta avvicinando sempre di più a ciò che fa PyTorch. Ogni funzione di base che dobbiamo differenziare è scritta come un oggetto "torch.autograd.Function" che ha un metodo "avanti" e uno "indietro". PyTorch manterrà quindi traccia di qualsiasi calcolo che facciamo per poter eseguire correttamente il passaggio all'indietro, a meno che non impostiamo l'attributo "requires_grad" dei nostri tensori su "False".

Scrivere uno di questi è (quasi) facile come scrivere le nostre lezioni originali. La differenza è che scegliamo cosa salvare e cosa inserire in una variabile di contesto (in modo da assicurarci di non salvare nulla di cui non abbiamo bisogno) e restituiamo i gradienti nel passaggio `backward`. È molto raro dover scrivere la propria "Funzione" ma se mai ti servisse qualcosa di esotico o volessi pasticciare con i gradienti di una funzione regolare, ecco come scriverne una:

In [ ]:
from torch.autograd import Function

class MyRelu(Function):
    @staticmethod
    def forward(ctx, i):
        result = i.clamp_min(0.)
        ctx.save_for_backward(i)
        return result
    
    @staticmethod
    def backward(ctx, grad_output):
        i, = ctx.saved_tensors
        return grad_output * (i>0).float()

La struttura utilizzata per costruire un modello più complesso che sfrutta queste "funzioni" è "torch.nn.Module". Questa è la struttura di base di tutti i modelli e di tutte le reti neurali che hai visto finora ereditate da quella classe. Aiuta soprattutto a registrare tutti i parametri addestrabili, che come abbiamo visto possono essere utilizzati nel ciclo di allenamento.

Per implementare un `nn.Module` devi solo:

- Assicurati che la superclasse `__init__` venga chiamata per prima quando la inizializzi.
- Definire eventuali parametri del modello come attributi con `nn.Parameter`.
- Definisci una funzione "avanti" che restituisca l'output del tuo modello.

Ad esempio, ecco il livello lineare da zero:

In [ ]:
import torch.nn as nn

class LinearLayer(nn.Module):
    def __init__(self, n_in, n_out):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(n_out, n_in) * sqrt(2/n_in))
        self.bias = nn.Parameter(torch.zeros(n_out))
    
    def forward(self, x): return x @ self.weight.t() + self.bias

Come vedi, questa classe tiene automaticamente traccia di quali parametri sono stati definiti:

In [ ]:
lin = LinearLayer(10,2)
p1,p2 = lin.parameters()
p1.shape,p2.shape

(torch.Size([2, 10]), torch.Size([2]))

È grazie a questa funzionalità di `nn.Module` che possiamo semplicemente dire `opt.step()` e avere un ottimizzatore che esegue un ciclo attraverso i parametri e li aggiorna ciascuno.

Tieni presente che in PyTorch i pesi sono archiviati come matrice `n_out x n_in`, motivo per cui abbiamo la trasposizione nel passaggio in avanti.

Utilizzando il livello lineare di PyTorch (che utilizza anche l'inizializzazione Kaiming), il modello che abbiamo costruito durante questo capitolo può essere scritto in questo modo:

In [ ]:
class Model(nn.Module):
    def __init__(self, n_in, nh, n_out):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(n_in,nh), nn.ReLU(), nn.Linear(nh,n_out))
        self.loss = mse
        
    def forward(self, x, targ): return self.loss(self.layers(x).squeeze(), targ)

fastai fornisce la propria variante di `Module` che è identica a `nn.Module`, ma non richiede di chiamare `super().__init__()` (lo fa automaticamente):

In [ ]:
class Model(Module):
    def __init__(self, n_in, nh, n_out):
        self.layers = nn.Sequential(
            nn.Linear(n_in,nh), nn.ReLU(), nn.Linear(nh,n_out))
        self.loss = mse
        
    def forward(self, x, targ): return self.loss(self.layers(x).squeeze(), targ)

Nell'ultimo capitolo, inizieremo da un modello di questo tipo e vedremo come costruire un ciclo di allenamento da zero e rifattorizzarlo in base a quello che abbiamo utilizzato nei capitoli precedenti.

## Conclusione

In questo capitolo abbiamo esplorato le basi del deep learning, iniziando con la moltiplicazione di matrici e passando all'implementazione da zero dei passaggi avanti e indietro di una rete neurale. Abbiamo quindi rifattorizzato il nostro codice per mostrare come funziona PyTorch dietro le quinte.

Ecco alcune cose da ricordare:

- Una rete neurale è fondamentalmente un insieme di moltiplicazioni di matrici con non linearità intermedie.
- Python è lento, quindi per scrivere codice veloce dobbiamo vettorizzarlo e sfruttare tecniche come l'aritmetica degli elementi e il broadcasting.
- Due tensori sono trasmissibili se le dimensioni che iniziano dalla fine e vanno all'indietro coincidono (se sono uguali, o uno di essi è 1). Per rendere i tensori trasmettebili, potrebbe essere necessario aggiungere dimensioni di dimensione 1 con l'indice "unsqueeze" o "None".
- Inizializzare correttamente una rete neurale è fondamentale per avviare l'addestramento. L'inizializzazione Kaiming dovrebbe essere utilizzata quando sono presenti non linearità ReLU.
- Il passaggio all'indietro è la regola a catena applicata più volte, calcolando i gradienti dall'output del nostro modello e tornando indietro, uno strato alla volta.
- Quando si crea una sottoclasse di `nn.Module` (se non si utilizza `Module` di fastai) dobbiamo chiamare il metodo della superclasse `__init__` nel nostro metodo `__init__` e dobbiamo definire una funzione `forward` che accetta un input e restituisce il risultato desiderato.

## Questionario

1. Scrivi il codice Python per implementare un singolo neurone.
1. Scrivi il codice Python per implementare ReLU.
1. Scrivi il codice Python per uno strato denso in termini di moltiplicazione di matrici.
1. Scrivi il codice Python per un livello denso in Python semplice (ovvero con la comprensione delle liste e le funzionalità integrate in Python).
1. Qual è la "dimensione nascosta" di un livello?
1. Cosa fa il metodo "t" in PyTorch?
1. Perché la moltiplicazione di matrici scritta in Python semplice è molto lenta?
1. In `matmul`, perché è `ac==br`?
1. In Jupyter Notebook, come si misura il tempo impiegato per l'esecuzione di una singola cella?
1. Cos'è l'"aritmetica elementare"?
1. Scrivi il codice PyTorch per verificare se ogni elemento di "a" è maggiore dell'elemento corrispondente di "b".
1. Cos'è un tensore di rango 0? Come lo converti in un semplice tipo di dati Python?
1. Cosa restituisce questo e perché? `tensore([1,2]) + tensore([1])`
1. Cosa restituisce questo e perché? `tensore([1,2]) + tensore([1,2,3])`
1. In che modo l'aritmetica elementare ci aiuta ad accelerare `matmul`?
1. Quali sono le regole di trasmissione?
1. Cos'è "expand_as"? Mostra un esempio di come può essere utilizzato per abbinare i risultati della trasmissione.
1. In che modo "unsqueeze" ci aiuta a risolvere alcuni problemi di trasmissione?
1. Come possiamo utilizzare l'indicizzazione per eseguire la stessa operazione di `unsqueeze`?
1. Come mostriamo il contenuto effettivo della memoria utilizzata per un tensore?
1. Quando si aggiunge un vettore di dimensione 3 a una matrice di dimensione 3×3, gli elementi del vettore vengono aggiunti a ciascuna riga o ciascuna colonna della matrice? (Assicurati di controllare la tua risposta eseguendo questo codice su un taccuino.)
1. La trasmissione e `expand_as` comportano un aumento dell'utilizzo della memoria? Perché o perché no?
1. Implementa "matmul" utilizzando la somma di Einstein.
1. Cosa rappresenta una lettera indice ripetuta sul lato sinistro di einsum?
1. Quali sono le tre regole della notazione delle somme di Einstein? Perché?
1. Quali sono il passaggio in avanti e il passaggio all'indietro di una rete neurale?
1. Perché dobbiamo memorizzare alcune delle attivazioni calcolate per gli strati intermedi nel passaggio in avanti?
1. Qual è lo svantaggio di avere attivazioni con una deviazione standard troppo lontana da 1?
1. In che modo l'inizializzazione del peso può aiutare a evitare questo problema?
1. Qual è la formula per inizializzare i pesi in modo tale da ottenere una deviazione standard di 1 per uno strato lineare semplice e per uno strato lineare seguito da ReLU?
1. Perché a volte dobbiamo utilizzare il metodo `squeeze` nelle funzioni di perdita?
1. Cosa fa l'argomento del metodo `squeeze`? Perché potrebbe essere importante includere questo argomento, anche se PyTorch non lo richiede?
1. Cos'è la "regola della catena"? Mostra l'equazione in una delle due forme presentate in questo capitolo.
1. Mostra come calcolare i gradienti di `mse(lin(l2, w2, b2), y)` utilizzando la regola della catena.
1. Qual è il gradiente di ReLU? Mostralo in matematica o in codice. (Non dovresti aver bisogno di memorizzarlo: prova a capirlo usando la tua conoscenza della forma della funzione.)
1. In quale ordine dobbiamo chiamare le funzioni `*_grad` nel passaggio all'indietro? Perché?
1. Cos'è "__call__"?
1. Quali metodi dobbiamo implementare quando scriviamo un `torch.autograd.Function`?
1. Scrivi `nn.Linear` da zero e verifica che funzioni.
1. Qual è la differenza tra `nn.Module` e `Module` di fastai?

### Ulteriori ricerche

1. Implementa ReLU come "torch.autograd.Function" e addestra un modello con esso.
1. Se sei portato per la matematica, scopri quali sono i gradienti di uno strato lineare in notazione matematica. Associatelo all'implementazione che abbiamo visto in questo capitolo.
1. Scopri il metodo "unfold" in PyTorch e utilizzalo insieme alla moltiplicazione di matrici per implementare la tua funzione di convoluzione 2D. Quindi addestra una CNN che lo utilizzi.
1. Implementa tutto in questo capitolo usando NumPy invece di PyTorch.